If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec12_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 12: Table Practice — Skyscrapers and Bike Sharing
v.ekc

No new table methods today — instead, the full toolkit on real data: finishing the skyscraper activity, then a deep dive into a Bay Area bike-share dataset. (Slides: KL Lecture 12 deck.)

**Today's sections:**
1. Skyscrapers, finished
2. Another join example
3. Bike sharing — distributions of durations
4. Start and end stations
5. Try It — fastest trips
6. Maps

In [ ]:
from datascience import *
%matplotlib inline
path_data = '../../../assets/data/'
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import numpy as np
import warnings
warnings.filterwarnings("ignore")


---
## 1. Skyscrapers, Finished

Question 1 is worked below — questions 2 and 3 are yours (again — no peeking at Lecture 11!).

In [ ]:
# From the CORGIS Dataset Project
# By Austin Cory Bart acbart@vt.edu
# Version 2.0.0, created 3/22/2016
# https://corgis-edu.github.io/corgis/csv/skyscrapers/

sky = Table.read_table('skyscrapers.csv')
sky = (sky.with_column('age', 2024 - sky.column('completed'))
          .drop('completed'))
sky.show(3)

1. For each city, what’s the tallest building for each material?

In [ ]:
# 1. For each city, what’s the tallest building for each material?

sky.pivot('material','city',values = 'height',collect = max)

2. For each city, what’s the height difference between the tallest steel building and the tallest concrete building?

In [ ]:
# 2. For each city, what’s the height difference between the tallest 
#    steel building and the tallest concrete building?



3. Generate a table of the names of the oldest buildings for each material for each city:

In [ ]:
# 3. Generate a table of the names of the oldest buildings for each 
#    material for each city:

# Hint: You can use sort to find the name of the oldest building in the dataset
sky.sort('age', descending=True).column('name').item(0)


# Put your solution here








<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Same pivot pattern as question 1 — different `values`/`collect`.*

```python
# 2.
tallest = sky.pivot('material', 'city', values='height', collect=max)
tallest.with_column(
    'difference',
    abs(tallest.column('steel') - tallest.column('concrete')))

# 3.
def first(s):
    return s.item(0)

sky.sort('completed').pivot('material', 'city', values='name', collect=first)
```

</details>

---
## 2. Another Join Example

Remember the census codes (0/1/2)? A tiny **code table** plus a join makes them readable — a very common real-world move.

In [ ]:
full = Table.read_table('nc-est2019-agesex-res.csv')
census = full.select('SEX', 'AGE', 'POPESTIMATE2019')
census.show(3)

In [ ]:
sex_codes = Table().with_columns(
    'SEX CODE', make_array(0, 1, 2),
    'CODE DEFINITION', make_array('All', 'Selected Male', 'Selected Female')
)
sex_codes

In [ ]:
census.join('SEX', sex_codes, 'SEX CODE').sort('AGE').show(6)

---
## 3. Bike Sharing 🚲

~350,000 bike trips in the SF Bay Area. Big data needs the histogram + filter combo:

In [ ]:
trip = Table.read_table('trip.csv')
trip.show(3)

### Distribution of Durations

The raw histogram is useless — a few *very* long rentals squash everything. Filter to commutes (under 30 min) first.

In [ ]:
trip.hist('Duration')

In [ ]:
trip.sort('Duration', descending=True)

In [ ]:
commute = trip.where('Duration', are.below(1800))
commute.hist('Duration')

In [ ]:
commute.hist('Duration', bins=np.arange(0, 1800, 250), unit='Second')

In [ ]:
# Approx percent of people who have 
# a ride duration between 250 and 500 seconds
# "between" = [250, 500) 

(500-250) * 0.15 

In [ ]:
commute.where('Duration', are.between(250, 500)).num_rows

In [ ]:
commute.num_rows

In [ ]:
129079 / 338343

> **Area check:** the `[250, 500)` bar has height ≈ 0.15 %/second × 250 seconds ≈ 38% — and counting rows directly gives 129,079/338,343 ≈ 38%. The Area Principle works. ✅

In [ ]:
commute.hist('Duration', bins=np.arange(0, 1800, 250), unit='Second')

In [ ]:
commute.hist('Duration', bins=60, unit='Second')

---
## 4. Start and End Stations

`group` for the most popular start, `pivot` for the full station-to-station grid.

In [ ]:
# Most common start station

starts = commute.group('Start Station').sort('count', descending=True)
starts

In [ ]:
# Number of trips between stations

commute.pivot('Start Station', 'End Station')

In [ ]:
# Average durations of trips between stations

commute.pivot('Start Station', 'End Station', values='Duration', collect=np.average)

---
### Try It 1: Fastest trips 😊

1. Find the fastest trip ever between each pair of stations.

2. Estimate the 5 stations closest to Civic Center BART by minimum trip time. *Hint: it's listed as `Civic Center BART (7th at Market)` — `are.containing` saves typing.*

In [ ]:
# Fill in the blank — replace the ... 😅
shortest = ...
shortest

In [ ]:
# Fill in the blank — replace the ... 😅
from_cc = ...
from_cc

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*"Fastest ever between each pair" is a pivot with `collect=min`; "closest to one station" filters first, then groups.*

```python
# 1.
shortest = trip.pivot('Start Station', 'End Station', values='Duration', collect=min)
shortest

# 2.
from_cc = (trip.where('Start Station', are.containing('Civic Center BART'))
               .select('End Station', 'Duration')
               .group('End Station', min)
               .sort('Duration min')
               .take(np.arange(5)))
from_cc
```

</details>

---
## 5. Maps 🗺️

Tables with `lat`/`long` columns can go straight onto a real map — `Marker.map_table` wants exactly (lat, long, labels).

In [ ]:
# Geographical data on the stations
stations = Table.read_table('station.csv').drop(4, 6)
stations

In [ ]:
sf_stations = stations.where('landmark', are.equal_to('San Francisco'))
sf_stations_map_data = (sf_stations
 .select('lat', 'long', 'name')
 .relabeled('name', 'labels'))
sf_stations_map_data.show(3)

In [ ]:
Marker.map_table(sf_stations_map_data)

### Try It 2 (Challenge): Map the neighborhood 🔥

1. Map all stations within 4 minutes (240 seconds) of minimum ride time from Civic Center BART. (`Marker.map_table?` below shows the docs.)

In [ ]:
Marker.map_table?

In [ ]:
# 1. stations within 4 minutes of Civic Center


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Chain it: filter trips from Civic Center → group by destination with `min` → keep under 240s → join with `stations` for coordinates → map.*

```python
# 1.
close = (trip.where('Start Station', are.containing('Civic Center BART'))
             .select('End Station', 'Duration')
             .group('End Station', min)
             .where('Duration min', are.below(240)))

close_map_data = (stations.join('name', close, 'End Station')
                          .select('lat', 'long', 'name')
                          .relabeled('name', 'labels'))
Marker.map_table(close_map_data)
```

</details>

---
> **Today's takeaway:** you just did an end-to-end analysis — load, filter, group, pivot, join, map — with nothing you didn't already know. Homework 5 and Lab 5 build from here.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — station-pair analysis
```python
t.pivot('cat1', 'cat2', values='v', collect=min)
t.where('cat', are.containing('text')).group('other_cat', min)
```